# Interactive self-eval

A two-phase loop over **one persistent game** and **one conversation**:

1. **Player phase.** Ask about the *current scene* — usually "what's the best
   move?", sometimes a general question ("are you facing the gold?"). The
   player answers in a **single generation**: one move token at most, and that
   move is **not applied yet**. It may run `[SEARCH <query>]` memory searches
   (semantic model + past reasoning only, settings-scrubbed) before answering.
2. **Analyst phase.** Control returns to you: the lower text box is pre-filled
   with the default analysis question — edit it or just press **Analyze**. The
   analyst reviews ONLY the player's latest reply, with privileged access (the
   exact frame **and its Settings JSON**), searches all memory tiers if it
   wants, and ends with a rating in [-1.0, 1.0]. Any `WRONG: "..."` error
   spans it emits are verified against the recorded player reply and rendered
   as highlights (unverified spans are flagged loudly instead). You can go
   **back and forth** with the analyst — each reply re-enables Analyze — and
   every verdict is stored in the same conversation, so the player sees past
   analyses in later rounds. When you're satisfied, press **Back to player
   (apply move)**: **only then** is the pending move propagated and the new
   board shown.

Buttons: **Ask** submits a player question; **Reset game** (player phase only —
same times Ask is available) swaps in a brand-new random bare board without
asking anything, and records that fact in the agent's memory; **Analyze**
submits a question to the analyst (repeatable within a round); **Back to
player (apply move)** ends the round and propagates the pending move;
**Restart conversation** starts a fresh thread + board.

In every text box, Shift+Enter inserts a plain newline (it never submits, and
never reaches Jupyter's run-cell shortcut) — buttons are the only submit
mechanism.

The **model dropdown** at the top of the panel switches between the registry
models (see `MODEL_CANDIDATES.md`), ordered by recommendation; switching is an
explicit **Switch model** press, and first use of a model downloads its
weights. Checking **"Save only one set of weights at a time"** makes a switch
restart the conversation and delete every other registry model's cached
weights from disk.

Prereqs (same as `play.ipynb`):
- Neo4j is up (`bash scripts/vast_neo4j_launch.sh`, or `docker compose up -d neo4j`).
- The semantic model is seeded once (`python -m agent.runner seed`).
- `bash scripts/setup_env.sh` (installs deps incl. `ipywidgets`, and downloads
  the spaCy + GLiNER weights NAMS' extraction pipeline needs).

The first cell connects NAMS and loads Gemma, so it can take a minute.

In [1]:
import os
import sys

# Run from the repo root so `agent` imports and `.env` resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from agent.self_eval_session import InteractiveSelfEvalSession

# Connects NAMS on a background loop and loads Gemma 4 E4B (first run is slow).
# Logging is on by default: every LLM generate call and every DB retrieval for
# this session is written under session.logger.run_dir.
session = InteractiveSelfEvalSession()
print("session_id:", session.session_id)
if session.logger is not None:
    print("log dir:", session.logger.run_dir)

/venv/main/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

session_id: 1802dd7dfeb146db820d0d35152dc9ca
log dir: logs/self_eval_2026-07-23_18-11-25


In [2]:
import html as _html

import ipywidgets as widgets
from IPython.display import HTML, Image, clear_output, display

from agent.notebook_ui import model_picker, tame_shift_enter
from agent.self_eval_session import (
    DEFAULT_ANALYST_QUESTION,
    DEFAULT_PLAYER_QUESTION,
)

# ----------------------------------------------------------- player controls
question_box = widgets.Textarea(
    value=DEFAULT_PLAYER_QUESTION,
    placeholder="Ask the player about the current scene (e.g. 'What is the best move?')...",
    description="You:",
    layout=widgets.Layout(width="600px", height="70px"),
)
ask_btn = widgets.Button(description="Ask", button_style="primary")
reset_btn = widgets.Button(description="Reset game", button_style="primary")
restart_btn = widgets.Button(description="Restart conversation", button_style="warning")
dump_btn = widgets.Button(description="Dump DB status", button_style="info")
out = widgets.Output()

# ---------------------------------------------------------- analyst controls
# Pre-filled with the default analysis question (tall enough to show all of
# it); edit it or submit as-is. Enabled only while a round is open. Analyze
# can be pressed repeatedly (multi-round chat); Back to player ends the round
# and propagates the pending move.
analyst_box = widgets.Textarea(
    value=DEFAULT_ANALYST_QUESTION,
    description="Analyst:",
    layout=widgets.Layout(width="600px", height="170px"),
)
analyze_btn = widgets.Button(description="Analyze", button_style="success")
back_btn = widgets.Button(description="Back to player (apply move)", button_style="primary")

# Frames are rendered at cfg.game_size (768x768 by default) but always shown
# at this fixed width so the conversation flow stays compact and scannable.
FRAME_WIDTH = 420  # px


def _sync_phase():
    """Enable exactly the controls of the current phase (no double-asks, no
    analyzing before playing). Reset game shares Ask's availability: player
    phase only, after the round has ended. Back to player is the analyst
    phase's exit."""
    player_on = session.phase == "player"
    question_box.disabled = not player_on
    ask_btn.disabled = not player_on
    reset_btn.disabled = not player_on
    analyst_box.disabled = player_on
    analyze_btn.disabled = player_on
    back_btn.disabled = player_on


def _show_frame(path, caption):
    if path:
        print(caption)
        display(Image(filename=path, width=FRAME_WIDTH))


def _show_searches(searches):
    for s in searches:
        print(f"  [SEARCH {s['query']}]")


def _show_player_result(result):
    _show_frame(result["before_path"], "-- frame the player saw --")
    _show_searches(result.get("searches", []))
    print(f"Player: {result['raw']}")
    if result["action"]:
        print(f"[pending move: {result['action']} -- NOT applied until the round ends]")
    elif result.get("bare_move"):
        print(
            f"[FORMAT ERROR: bare '{result['bare_move']}' without brackets -- "
            f"NOT a move, nothing will be propagated]"
        )
    else:
        print(
            "[WARNING: no move token detected. "
            "If a move was intended, this is a Format Error]"
        )
    if result.get("missing_think_close"):
        print(
            "[FORMAT ERROR: thinking model never closed its think block -- "
            "full raw text kept visible; the analyst has been told]"
        )
    print("\n>>> Analyst phase: edit or submit the question in the lower box.")
    print(">>> You can Analyze repeatedly; press Back to player when satisfied.")


def _show_wrong_spans(player_raw, wrong_spans):
    """Render the player reply with harness-VERIFIED wrong spans highlighted;
    unverified spans (words the player never wrote) get a loud warning."""
    verified = wrong_spans.get("verified", [])
    unverified = wrong_spans.get("unverified", [])
    if verified:
        marked = _html.escape(player_raw)
        for span in verified:
            marked = marked.replace(
                _html.escape(span),
                "<mark>" + _html.escape(span) + "</mark>",
            )
        print("\n-- player reply with VERIFIED wrong spans highlighted --")
        display(HTML(
            "<div style='white-space: pre-wrap; border-left: 3px solid #c00; "
            "padding-left: 8px;'>" + marked + "</div>"
        ))
    if unverified:
        print("\n!!! UNVERIFIED wrong spans (not found verbatim in the player reply):")
        for span in unverified:
            print(f'    WRONG: "{span}"')


def _show_analyst_step(step):
    """Render each analyst generation as it happens (search calls included)."""
    if step["search_query"] is not None:
        print(f"Analyst: {step['text']}")
        print(f"  [running SEARCH: {step['search_query']}]")
    else:
        print(f"Analyst: {step['text']}")
    if step.get("missing_think_close"):
        print(
            "[FORMAT ERROR: the analyst model never closed its think block -- "
            "full raw text kept visible]"
        )


# Whether the current round already has at least one analyst reply (an empty
# box is only meaningful for the round's FIRST Analyze, where it means "use
# the default question").
round_has_analysis = False


def _ask_player(q):
    global round_has_analysis
    print(f"\n=== You: {q}")
    result = session.ask_player(q)
    _show_player_result(result)
    # Restore the default for the next player turn; analyst box gets its
    # default for the round's first Analyze (follow-ups start empty).
    question_box.value = DEFAULT_PLAYER_QUESTION
    analyst_box.value = DEFAULT_ANALYST_QUESTION
    round_has_analysis = False


def on_ask(_):
    q = question_box.value.strip()
    if not q:
        return
    ask_btn.disabled = reset_btn.disabled = True
    try:
        with out:
            _ask_player(q)
    finally:
        _sync_phase()


def on_reset(_):
    """Swap in a new board only -- does not submit a player question."""
    ask_btn.disabled = reset_btn.disabled = True
    try:
        with out:
            info = session.reset_game()  # recorded in the agent's memory
            print("\n*** Game reset: brand-new random board. ***")
            _show_frame(info["frame_path"], "-- board after reset --")
    finally:
        _sync_phase()


def on_analyze(_):
    """One analyst exchange; the round stays open, so Analyze re-enables and
    you can keep the conversation going before applying the move."""
    global round_has_analysis
    q = analyst_box.value.strip()
    if not q and round_has_analysis:
        return  # empty follow-up: ignore the stray click, no generation burned
    analyze_btn.disabled = back_btn.disabled = True
    try:
        with out:
            print(f"\n=== Analyst question: {q or DEFAULT_ANALYST_QUESTION}")
            result = session.ask_analyst(q, on_step=_show_analyst_step)
            round_has_analysis = True
            _show_wrong_spans(result["player_raw"], result["wrong_spans"])
            print("\n>>> Follow up with the analyst, or press Back to player to apply the move.")
        analyst_box.value = ""  # clear for follow-ups
    finally:
        _sync_phase()


def on_back(_):
    """End the round: propagate the pending move and hand control back to the
    player phase."""
    analyze_btn.disabled = back_btn.disabled = True
    try:
        with out:
            result = session.end_round()
            if result["action"]:
                print(
                    f"\n[move {result['action']} propagated  "
                    f"gold_collected={result['gold_collected']}  "
                    f"gold_remaining={result['gold_remaining']}]"
                )
                _show_frame(result["frame_path"], "-- board after the propagated move --")
                if result["gold_remaining"] == 0:
                    print("*** Gold collected! Reset the game to keep playing. ***")
            else:
                print("\n[no pending move to propagate]")
            print("\n>>> Player phase: ask the next question in the upper box.")
    finally:
        _sync_phase()


def on_restart(_):
    global round_has_analysis
    info = session.restart()
    with out:
        clear_output()
        print(f"New conversation. session_id={info['session_id']}")
        _show_frame(info["frame_path"], "-- fresh board --")
    question_box.value = DEFAULT_PLAYER_QUESTION
    analyst_box.value = DEFAULT_ANALYST_QUESTION
    round_has_analysis = False
    _sync_phase()


def on_dump(_):
    dump_btn.disabled = True
    try:
        with out:
            info = session.dump_db()
            print(f"\nDB dumped -> {info['path']}")
            print(f"  nodes = {info['nodes']}   relationships = {info['relationships']}")
            if session.logger is not None:
                print(f"  llm log: {session.logger.llm_txt}")
                print(f"  db  log: {session.logger.db_txt}")
    finally:
        dump_btn.disabled = False


ask_btn.on_click(on_ask)
reset_btn.on_click(on_reset)
analyze_btn.on_click(on_analyze)
back_btn.on_click(on_back)
restart_btn.on_click(on_restart)
dump_btn.on_click(on_dump)


def _on_model_switched(info):
    """Refresh the panel after a model switch. With the one-weights checkbox
    the session was fully restarted (new conversation, player phase); without
    it the round state is untouched and the exchange simply continues."""
    global round_has_analysis
    with out:
        if info["restarted"]:
            clear_output()
            print(f"New conversation under {info['label']}. "
                  f"session_id={session.session_id}")
            _show_frame(session.current_frame_path(), "-- fresh board --")
        else:
            print(f"\n[model switched to {info['label']} -- conversation continues]")
    if info["restarted"]:
        question_box.value = DEFAULT_PLAYER_QUESTION
        analyst_box.value = DEFAULT_ANALYST_QUESTION
        round_has_analysis = False
    _sync_phase()


# Model picker on top, then conversation, then input boxes: after a long
# exchange the controls sit right where you finished reading, no scrolling
# back up.
display(widgets.VBox([
    model_picker(session, on_switched=_on_model_switched),
    out,
    question_box,
    widgets.HBox([ask_btn, reset_btn, restart_btn, dump_btn]),
    analyst_box,
    widgets.HBox([analyze_btn, back_btn]),
]))
# Shift+Enter = plain newline in both boxes (never submits, never re-runs the cell).
tame_shift_enter(question_box, analyst_box)

_sync_phase()
# Show the current starting board.
with out:
    print(f"session_id={session.session_id}")
    _show_frame(session.current_frame_path(), "-- starting board --")

### Restore the DB to "semantic seeding only" (optional cleanup)

Same escape hatch as in `play.ipynb`: deletes **all** episodic nodes
(`Conversation`, `Message`, `GameSnapshot`, `ReasoningTrace`/`ReasoningStep`)
and keeps only the seeded semantic model (`Entity` + `Preference` — including
any tips saved via `[WRITE_TIP]`, so wipe those manually if unwanted). Gated by
`if False:` so it never runs by accident.

In [ ]:
# Flip to `if True:` to wipe episodic memory and restore the seeded state.
if False:
    import shutil

    deleted = session.reset_memory_to_seed()
    print("deleted nodes:", deleted)

    img_dir = session.cfg.image_dir
    if img_dir.exists():
        shutil.rmtree(img_dir, ignore_errors=True)
        print("cleared image dir:", img_dir)

    info = session.restart()
    print("new session_id:", info["session_id"])

When you're done, release the model and close the memory client:

In [ ]:
session.close()